# ORBIT PULSE - Conjunction Risk Intelligence Exploration
### SW-5 Satellite Conjunction Risk Intelligence & Decision Support Prototype
**Single Source of Truth**: Follows `ORBIT PULSE Final Prototype Master Blueprint`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from src.data_loader import load_raw_cdm_data, get_thresholds_config
from src.validation import validate_cdm_records
from src.event_builder import build_event_histories
from src.snapshot_48h import build_all_snapshots
from src.feature_engineering import build_dataset_feature_matrix
from src.predict import OrbitPulsePredictor

print("ORBIT PULSE packages loaded successfully.")

## 1. Data Ingestion & Schema Validation

In [ ]:
raw_df = load_raw_cdm_data('../data/raw/synthetic_cdm_fixture.csv')
valid_df, quarantine_df, audit = validate_cdm_records(raw_df)
print(f"Raw Ingested: {len(raw_df)} | Valid CDMs: {len(valid_df)} | Quarantined: {len(quarantine_df)}")
print(f"Duplicates Detected: {audit['duplicate_records_count']}")

## 2. Event Histories & TCA-48h Cutoff Snapshots

In [ ]:
events = build_event_histories(valid_df)
snapshots = build_all_snapshots(events)
print(f"Total Events: {len(events)}")
eligible_count = sum(1 for s in snapshots.values() if s.eligible)
print(f"Eligible Events (<= TCA-48h): {eligible_count}")
print(f"Ineligible Events: {len(snapshots) - eligible_count}")

## 3. End-to-End Prediction & Evidence Intelligence

In [ ]:
predictor = OrbitPulsePredictor(model_path='../models/risk_model.joblib', metadata_path='../models/model_metadata.json')
sample_event_id = list(snapshots.keys())[0]
sample_snap = snapshots[sample_event_id]
pred_out = predictor.predict_snapshot(sample_snap)
print(f"Event: {sample_snap.event_id}")
print(f"Advisory Status: {pred_out['advisory_label']}")
print(f"Predicted Final Pc: {pred_out['predicted_value']:.2e}")
print(f"Freshness: {pred_out['evidence_intelligence']['freshness']['category']}")
print(f"Completeness: {pred_out['evidence_intelligence']['completeness']['category']}")
print(f"Stability: {pred_out['evidence_intelligence']['stability']['stability_status']}")